In [1]:
import asyncio
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from pydantic import BaseModel, Field
from dataclasses import dataclass
from typing import Optional

# ── Config ────────────────────────────────────────────────────────────────────
VLLM_BASE_URL = "http://localhost:8000/v1"
VLLM_MODEL    = "Qwen2.5-7B-Instruct"
API_KEY       = "abc-123"

# ── Deps ──────────────────────────────────────────────────────────────────────
@dataclass
class ComplaintDeps:
    complaint_id:   str
    complaint_text: str
    channel:        str
    account_type:   str

# ── Output schema ─────────────────────────────────────────────────────────────
class ClassificationResult(BaseModel):
    category:      str = Field(description="fraud | card_dispute | kyc_onboarding | loan_emi | net_banking_upi | branch_atm | insurance | general")
    priority:      str = Field(description="P1 | P2 | P3")
    sentiment:     str = Field(description="angry | distressed | neutral")
    pii_detected:  bool
    summary:       str = Field(description="max 10 words")
    reasoning:     str = Field(description="one sentence explaining decision")

# ── Agent ─────────────────────────────────────────────────────────────────────
provider = OpenAIProvider(base_url=VLLM_BASE_URL, api_key=API_KEY)
model    = OpenAIChatModel(VLLM_MODEL, provider=provider)

agent = Agent(
    model=model,
    output_type=ClassificationResult,
    deps_type=ComplaintDeps,
    system_prompt="""You are a BFSI complaint routing agent for an Indian bank.
Classify complaints and return structured JSON.

Categories: fraud, card_dispute, kyc_onboarding, loan_emi, net_banking_upi, branch_atm, insurance, general
Priority: P1 (fraud/critical), P2 (card blocked/KYC/UPI), P3 (everything else)
Sentiment: detect from tone — angry, distressed, neutral
""",
)


To Test Agent work run below cell. Sentiment & Priority detection. ( Need to remove later on)


In [6]:
async def test_single():
    complaint_text = "My HDFC credit card was charged Rs 45000 at a foreign merchant. I am in Mumbai right now. Block my card immediately."

    deps = ComplaintDeps(
        complaint_id="test-001",
        complaint_text=complaint_text,
        channel="web",
        account_type="credit_card",
    )

    result = await agent.run(
        f"Classify this complaint:\n\n{complaint_text}",
        deps=deps,
    )

    print("── Input ─────────────────────────────────")
    print(complaint_text)
    print("\n── Agent Output ──────────────────────────")
    print(f"category:     {result.output.category}")
    print(f"priority:     {result.output.priority}")
    print(f"sentiment:    {result.output.sentiment}")
    print(f"pii_detected: {result.output.pii_detected}")
    print(f"summary:      {result.output.summary}")
    print(f"reasoning:    {result.output.reasoning}")

await test_single()


── Input ─────────────────────────────────
My HDFC credit card was charged Rs 45000 at a foreign merchant. I am in Mumbai right now. Block my card immediately.

── Agent Output ──────────────────────────
category:     fraud
priority:     P1
sentiment:    angry
pii_detected: True
summary:      Card charged abroad, urgent block request
reasoning:    Transaction occurred in a different country than the reported location, indicative of potential fraud. Immediate action required.


In [7]:
""" To test one complaint per category to check if correct classification is done """

import json

async def test_all_categories():
    with open("complaints.json") as f:
        complaints = json.load(f)

    # one complaint per category
    seen = set()
    samples = []
    for c in complaints:
        if c["category"] not in seen:
            seen.add(c["category"])
            samples.append(c)
        if len(seen) == 8:
            break

    print(f"Testing {len(samples)} complaints...\n")

    correct = 0
    for c in samples:
        deps = ComplaintDeps(
            complaint_id=c["id"],
            complaint_text=c["text"],
            channel=c["channel"],
            account_type=c["account_type"],
        )

        result = await agent.run(
            f"Classify this complaint:\n\n{c['text']}",
            deps=deps,
        )

        match = "✓" if result.output.category == c["category"] else "✗"
        if result.output.category == c["category"]:
            correct += 1

        print(f"{match} [{c['category']:<20}] → [{result.output.category:<20}] {result.output.priority} {result.output.sentiment}")
        print(f"   {c['text'][:70]}...")
        print()

    print(f"Accuracy: {correct}/{len(samples)} = {correct/len(samples)*100:.0f}%")

await test_all_categories()

Testing 8 complaints...

✓ [card_dispute        ] → [card_dispute        ] P1 distressed
   My HDFC credit card was declined at an ATM for an unauthorized transac...

✓ [fraud               ] → [fraud               ] P1 angry
   My SBI debit card was debited Rs 25,000 from an unknown location this ...

✗ [general             ] → [kyc_onboarding      ] P3 neutral
   The ATM card I applied for two months ago has still not been delivered...

✓ [kyc_onboarding      ] → [kyc_onboarding      ] P3 neutral
   My KYC process has been stuck for over a month now. I submitted all th...

✓ [insurance           ] → [insurance           ] P3 neutral
   I made a lump sum claim under my term insurance policy in March, but I...

✓ [loan_emi            ] → [loan_emi            ] P3 neutral
   EMI for my business loan of Rs 25000 bounced this month despite having...

✓ [branch_atm          ] → [branch_atm          ] P3 neutral
   ATM at Ghatkopar East is giving me an error message 'Insufficient Bala...

✓